# Garbage Classification - CNN

10 sınıflı çöp sınıflandırma. Üç model karşılaştırılır: ResNet50 (transfer learning + fine-tuning), MobileNetV2 (transfer learning) ve sıfırdan CNN.

**Not:** Her hücreyi sırayla çalıştır. Eğitim uzun sürebilir; eğitilen modeller `models/` klasörüne kaydedilir.

In [ ]:
import os

import tensorflow as tf
from tensorflow.keras import layers

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 1. Veri Setini Yükle

In [ ]:
# Veriler data/original altinda 10 sinif klasoru halinde.
dataset_path = "data/original"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print("Siniflar:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

## 2. Sınıf Ağırlıkları (dengesizlik için)

`trash` (453) ile `clothes` (1892) arasinda ~4 kat fark var. `class_weight` ile az ornekli siniflar dengelenir.

In [ ]:
counts = [len(os.listdir(os.path.join(dataset_path, c))) for c in class_names]
total = sum(counts)
class_weight = {i: total / (num_classes * counts[i]) for i in range(num_classes)}

for c, n, w in zip(class_names, counts, class_weight.values()):
    print(f"{c:12s} n={n:5d}  weight={w:.3f}")

## 3. ResNet50 — Transfer Learning

**Önemli:** ImageNet agirlikli ResNet50 kendi `preprocess_input` fonksiyonunu bekler (basit `/255` degil). Bu yuzden modelin icine `resnet50.preprocess_input` gomuldu.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

resnet_base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
resnet_base.trainable = False

# Fonksiyonel API: Grad-CAM icin ic katmanlara erisilebilsin diye.
inputs = tf.keras.Input(shape=(224, 224, 3))
x = layers.Lambda(resnet_preprocess, name="resnet_preprocess")(inputs)
x = resnet_base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
resnet_model = tf.keras.Model(inputs, outputs, name="resnet50_model")

resnet_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

resnet_history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight,
)

### 3b. Fine-tuning (son 30 katman)

In [ ]:
resnet_base.trainable = True
for layer in resnet_base.layers[:-30]:
    layer.trainable = False

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

resnet_ft_history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=class_weight,
)

os.makedirs("models", exist_ok=True)
resnet_model.save("models/resnet50_finetuned.keras")

## 4. MobileNetV2 — Transfer Learning

MobileNetV2 girdiyi `[-1, 1]` araliginda bekler; kendi `preprocess_input`'i kullanildi.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

mobilenet_base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
mobilenet_base.trainable = False

mobilenet_model = tf.keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Lambda(mobilenet_preprocess, name="mobilenet_preprocess"),
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
])

mobilenet_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

mobilenet_history = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight,
)

mobilenet_model.save("models/mobilenetv2.keras")

## 5. Sıfırdan CNN

Onceden egitilmemis basit CNN; burada `Rescaling(1./255)` uygun.

In [ ]:
scratch_model = tf.keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax"),
])

scratch_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

scratch_history = scratch_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight,
)

scratch_model.save("models/scratch_cnn.keras")

## 6. Değerlendirme ve Karşılaştırma

In [ ]:
resnet_loss, resnet_acc = resnet_model.evaluate(val_ds)
mobilenet_loss, mobilenet_acc = mobilenet_model.evaluate(val_ds)
scratch_loss, scratch_acc = scratch_model.evaluate(val_ds)

results = pd.DataFrame({
    "Model": ["ResNet50", "MobileNetV2", "Scratch CNN"],
    "Validation Accuracy": [resnet_acc, mobilenet_acc, scratch_acc],
    "Validation Loss": [resnet_loss, mobilenet_loss, scratch_loss],
})
print(results)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results["Model"], results["Validation Accuracy"])
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

### 6b. En iyi model için confusion matrix + classification report

In [ ]:
# En yuksek dogrulukta modeli sec
best_model = max(
    [(resnet_acc, resnet_model), (mobilenet_acc, mobilenet_model), (scratch_acc, scratch_model)],
    key=lambda t: t[0],
)[1]
print("En iyi model:", best_model.name)

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm_matrix, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=90)
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## 7. Grad-CAM (ResNet50)

Modelin karari verirken goruntunun hangi bolgesine baktigini gosterir. Ic ice gomulu `resnet_base` icin iki asamali (conv + classifier) yaklasim kullanilir; boylece graf kopmasi yasanmaz.

In [ ]:
last_conv_layer_name = "conv5_block3_out"

# 1) input -> son konvolusyon ciktisi
conv_model = tf.keras.Model(
    resnet_base.input,
    resnet_base.get_layer(last_conv_layer_name).output,
)

# 2) son konvolusyon ciktisi -> tahmin (resnet_base sonrasi katmanlar)
classifier_input = tf.keras.Input(shape=resnet_base.get_layer(last_conv_layer_name).output.shape[1:])
x = classifier_input
for layer in resnet_model.layers[3:]:  # Input, Lambda, resnet_base sonrasi
    x = layer(x)
classifier_model = tf.keras.Model(classifier_input, x)


def make_gradcam_heatmap(img_array):
    preprocessed = resnet_preprocess(tf.identity(img_array))
    with tf.GradientTape() as tape:
        conv_outputs = conv_model(preprocessed)
        tape.watch(conv_outputs)
        predictions = classifier_model(conv_outputs)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

In [ ]:
# val setinden bir goruntu al
for image_batch, label_batch in val_ds.take(1):
    image = image_batch[0]
    break

img_array = tf.expand_dims(image, axis=0)
heatmap, pred_index = make_gradcam_heatmap(img_array)
print("Tahmin:", class_names[pred_index])

img = image.numpy().astype("uint8")
heatmap_uint8 = np.uint8(255 * heatmap)

jet = cm.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[heatmap_uint8]
jet_heatmap = tf.keras.utils.array_to_img(jet_heatmap)
jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
jet_heatmap = tf.keras.utils.img_to_array(jet_heatmap)

superimposed_img = np.clip(jet_heatmap * 0.4 + img, 0, 255).astype("uint8")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original Image")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(superimposed_img)
plt.title(f"Grad-CAM: {class_names[pred_index]}")
plt.axis("off")
plt.show()